# Historical baseline: rentals from the previous week

Predicts hourly rentals per station from **past rentals only**:
- `last_week` - same station, same hour, **7 days earlier**
- `avg_4w` - average of the same hour over the **previous 4 weeks**

No training. Scored on the train+validation period (to see how it behaves while you're choosing models) and on the test period (generalisation only). Each is shown on **all rows** and on **actual > 0 only**.

## 1. Connect to GitHub and load the data

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/amelia-lee-smu/CS610-Applied-Machine-Learning.git"
BRANCH   = "main"     # README links the processed data on branch "feat/eda-for-proposal";
                      # if data/processed is missing on main, change this, delete the cloned folder, re-run
REPO_DIR = Path("CS610-Applied-Machine-Learning")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])

print("Repo folders:", sorted(p.name for p in REPO_DIR.iterdir() if p.is_dir() and not p.name.startswith(".")))

In [ ]:
try:
    import pyarrow  # noqa
except ImportError:
    %pip install -q pyarrow

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_poisson_deviance

DATA_DIR = REPO_DIR / "data" / "processed"
PRED_DIR = REPO_DIR / "data" / "predictions"

# column names of data/processed/*.parquet
STATION, TARGET, DATE, HOUR = "start station name", "num_rentals", "date", "hour_of_day"

def read(p):
    return pd.read_parquet(p) if Path(p).suffix == ".parquet" else pd.read_csv(p)

def find(name):
    hits = sorted(DATA_DIR.rglob(name)) if DATA_DIR.exists() else []
    if not hits:
        raise FileNotFoundError(f"{name} not found under {DATA_DIR}. Try another BRANCH in the first cell "
                                f"(e.g. 'feat/eda-for-proposal'), delete the cloned folder and re-run.")
    p = hits[0]
    if p.stat().st_size < 300 and b"git-lfs" in p.read_bytes():
        raise RuntimeError(f"{p.name} is a Git LFS pointer - run `git lfs pull` in the repo.")
    return p

def prep(df):
    df = df.copy().reset_index(drop=True)          # keep the original row order (prediction files rely on it)
    df["_ts"] = pd.to_datetime(df[DATE]) + pd.to_timedelta(df[HOUR], unit="h")
    return df

trainval = prep(read(find("train_validation.parquet")))     # 1 Jan 2020 - 28 Feb 2021 (CV folds live here)
test     = prep(read(find("test.parquet")))                 # 1 Mar - 30 Apr 2021
KEYS = [STATION, "_ts"]
history = pd.concat([trainval, test])[KEYS + [TARGET]].drop_duplicates(KEYS)

print(f"train+validation: {trainval.shape}  {trainval['_ts'].min()} -> {trainval['_ts'].max()}")
print(f"test:             {test.shape}  {test['_ts'].min()} -> {test['_ts'].max()}")
print(f"zero-rental hours:  train+val {(trainval[TARGET] == 0).mean():.1%}   test {(test[TARGET] == 0).mean():.1%}")

## 2. Baseline predictions

In [ ]:
def add_baselines(df, weeks=(1, 2, 3, 4)):
    """Historical baselines for every row of df (row order preserved).
    last_week = same station & hour 7 days earlier; avg_4w = mean of the previous 4 weeks (same hour)."""
    out = df[KEYS].copy()
    for w in weeks:
        lag = history.copy()
        lag["_ts"] = lag["_ts"] + pd.Timedelta(weeks=w)       # line history up with the hour being predicted
        out = out.merge(lag.rename(columns={TARGET: f"lag_{w}w"}), on=KEYS, how="left")
    out["last_week"] = out["lag_1w"]
    out["avg_4w"] = out[[f"lag_{w}w" for w in weeks]].mean(axis=1)
    return out[["last_week", "avg_4w"]]

In [ ]:
bv, bt = add_baselines(trainval), add_baselines(test)
ok_v, ok_t = bv.notna().all(axis=1), bt.notna().all(axis=1)       # the first weeks of 2020 have no history
print(f"rows with history: train+val {ok_v.sum():,}/{len(trainval):,}   test {ok_t.sum():,}/{len(test):,}")

## 3. Evaluate

In [ ]:
EPS = 1e-3     # Poisson deviance needs predictions > 0 ("0 rentals last week" would give an infinite deviance)

def metrics(y, p):
    p = np.clip(np.asarray(p, float), EPS, None)
    return {"Poisson dev": mean_poisson_deviance(y, p), "MAE": mean_absolute_error(y, p),
            "RMSE": mean_squared_error(y, p) ** 0.5}

def evaluate(y, preds, groups=None, nonzero=False):
    """Mean of per-group metrics (groups = CV folds; None = one group). nonzero=True keeps only actual > 0."""
    y = np.asarray(y, float)
    groups = np.zeros(len(y), int) if groups is None else np.asarray(groups)
    rows = {}
    for name, p in preds.items():
        p = np.asarray(p, float); ms = []
        for g in np.unique(groups):
            m = (groups == g) & ((y > 0) if nonzero else True)
            ms.append(metrics(y[m], p[m]))
        rows[name] = pd.DataFrame(ms).mean()
    return pd.DataFrame(rows).T

In [ ]:
cols = ["last_week", "avg_4w"]
yv, yt = trainval.loc[ok_v, TARGET], test.loc[ok_t, TARGET]
out = {}
for label, nz in [("ALL rows", False), ("actual > 0 only", True)]:
    out[("train+val", label)] = evaluate(yv, {c: bv.loc[ok_v, c] for c in cols}, nonzero=nz)
    out[("test", label)]      = evaluate(yt, {c: bt.loc[ok_t, c] for c in cols}, nonzero=nz)
pd.concat(out, names=["period", "subset"]).round(3)

Choose between `last_week` and `avg_4w` on the **train+val** rows; the **test** rows are only there to check the choice generalises.

## 4. Save predictions
Saved with their keys (date, hour, station) so other notebooks can match them to rows. Commit them from your own checkout.

In [ ]:
SAVE = True
if SAVE:
    PRED_DIR.mkdir(parents=True, exist_ok=True)
    for c in cols:
        for tag, d, b in [("validation", trainval, bv), ("test", test, bt)]:
            o = d[[DATE, HOUR, STATION]].copy(); o["prediction"] = b[c].to_numpy()
            o.to_parquet(PRED_DIR / f"{tag}_baseline_{c}.parquet", index=False)
    print("saved:", sorted(p.name for p in PRED_DIR.glob("*_baseline_*")))